### Imports

In [1]:
import sys
import os

sys.path.append(os.path.abspath("..")) 

import json
import pandas as pd
from datetime import datetime
import re
from typing import Optional
from tqdm.auto import tqdm

### Helper functions for cleaning

In [5]:
def parse_date(date_str: str) -> str:
    """Parse date string handling multiple formats and languages"""
    if not date_str:
        raise Exception("No date str found")
    
    # Spanish to English month mapping
    spanish_months = {
        'enero': 'January', 'febrero': 'February', 'marzo': 'March',
        'abril': 'April', 'mayo': 'May', 'junio': 'June',
        'julio': 'July', 'agosto': 'August', 'septiembre': 'September',
        'octubre': 'October', 'noviembre': 'November', 'diciembre': 'December'
    }
    
    # Replace Spanish month names with English
    date_lower = date_str.lower()
    for spanish, english in spanish_months.items():
        if spanish in date_lower:
            date_str = date_str.replace(spanish.capitalize(), english)
            date_str = date_str.replace(spanish, english)
            break
    
    # Try to parse the date
    date_formats = ['%B %d, %Y', '%b %d, %Y', '%Y-%m-%d']
    
    for fmt in date_formats:
        try:
            date_obj = datetime.strptime(date_str, fmt)
            return date_obj.isoformat()
        except ValueError:
            continue

    raise Exception(f"Failed to parse {date_str}")

def clean_text(text: str | None) -> str | None:
    """Normalize whitespace and line breaks in text"""
    if not text:
        return text

    # Replace multiple spaces with single space
    text = re.sub(r'\s+', ' ', text)
    # Normalize line breaks
    text = text.replace('\r\n', '\n').replace('\r', '\n')
    # Remove leading/trailing whitespace
    text = text.strip()
    
    return text

def standardize_verdict(verdict: Optional[str]) -> Optional[str]:
    """
    Standardize verdict labels
    Maps Politifact verdicts to consistent format
    """
    if not verdict:
        return None
    
    verdict_map = {
        'true': 'true',
        'mostly-true': 'mostly-true',
        'half-true': 'half-true',
        'barely-true': 'mostly-false',
        'mostly-false': 'mostly-false',
        'false': 'false',
        'pants-fire': 'false',
        'pants-on-fire': 'false'
    }
    
    verdict_lower = verdict.lower().strip()
    return verdict_map.get(verdict_lower, verdict_lower)

def clean_politifact_article(article: dict) -> dict:
    """Clean a single Politifact article"""
    cleaned = article.copy()
    
    # Copy fields directly (already at top level)
    cleaned['title'] = clean_text(article.get('title'))
    cleaned['content'] = clean_text(article.get('content'))
    cleaned['claim'] = clean_text(article.get('claim'))
    cleaned['verdict'] = standardize_verdict(article.get('verdict'))
    cleaned['authors'] = article.get('authors', [])
    
    # Convert date to ISO string
    cleaned['publish_date'] = parse_date(article['publish_date'])
    
    # Add source bias
    cleaned["source_bias"] = "LEFT-CENTER"
    
    return cleaned

### Load in dataset

In [14]:
with open('../outputs/politifact-metadata-updated_progress.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

### Clean 

In [15]:
cleaned_data = [clean_politifact_article(article) for article in tqdm(data, desc="Cleaning articles")]

Cleaning articles:   0%|          | 0/20536 [00:00<?, ?it/s]

### Convert to DF and inspect

In [16]:
df = pd.DataFrame(cleaned_data)

print(f"Total articles: {len(cleaned_data)}")
print(f"\nVerdict distribution:")
print(df['verdict'].value_counts())
print(f"\nSample of cleaned data:")
print(df[['title', 'publish_date', 'verdict']].head())
print(f"\nArticles with missing dates: {df['publish_date'].isna().sum()}")
print(f"Articles with no authors: {df['authors'].apply(len).eq(0).sum()}")

Total articles: 20536

Verdict distribution:
verdict
false           12157
mostly-false     2664
half-true        2300
mostly-true      2021
true             1083
full-flop         288
half-flip          20
no-flip             3
Name: count, dtype: int64

Sample of cleaned data:
                                               title         publish_date  \
0  “I have just gotten the highest poll numbers o...  2025-11-24T00:00:00   
1  “There’s about 1,400 criminal illegal aliens t...  2025-11-24T00:00:00   
2  West Virginia is “the only state losing popula...  2025-11-21T00:00:00   
3  A pro-Donald Trump Montana town planned a “ped...  2025-11-21T00:00:00   
4  “We’re not cutting science. We’re not cutting ...  2025-11-20T00:00:00   

  verdict  
0   false  
1   false  
2   false  
3   false  
4   false  

Articles with missing dates: 0
Articles with no authors: 6


### Save cleaned data to separate JSON files

In [17]:
output_dir = '../outputs_clean/politifact'
os.makedirs(output_dir, exist_ok=True)

with open(f'{output_dir}/politifact_cleaned_final.json', 'w', encoding='utf-8') as f:
    json.dump(cleaned_data, f, indent=2, ensure_ascii=False)

print("Cleaning complete! File saved.")

Cleaning complete! File saved.
